# 01 — RAG Foundations: Evidence Before Answers

**Track:** Beginner · **Stage:** Foundation

NovaTech wants an Enterprise Knowledge Assistant that can answer questions over finance reviews, HR policies, IT runbooks, project docs, and vendor contracts. In this first notebook, we build the basic Retrieval-Augmented Generation (RAG) loop using **LangChain**, one of the most prominent enterprise AI SDKs.

## What you will build

- A basic LangChain RAG pipeline.
- A demonstration of naive generation vs. grounded generation.
- An inspectable trace of retrieved evidence.

## Setup: Model and SDK

For this module, we use LangChain. We will use a mock LLM and mock embeddings for deterministic, zero-cost local execution, but the API calls perfectly mirror production usage of `ChatOpenAI`.

In [ ]:
# !pip install langchain langchain-core langchain-community

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_community.llms.fake import FakeListLLM
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_community.embeddings import FakeEmbeddings

## 1. The Naive Approach (No RAG)

First, let's ask a question without providing any external context. The model will either hallucinate or state it doesn't know.

In [ ]:
# We use a FakeListLLM to simulate an LLM's hallucination for this exercise.
# In production, you would use: from langchain_openai import ChatOpenAI; llm = ChatOpenAI()
mock_hallucinated_responses = [
    "NovaTech's Q2 2025 revenue increased by 5%."
]
llm = FakeListLLM(responses=mock_hallucinated_responses)

question = "What increased by 14% at NovaTech in Q2 2025?"
print("Model answer without RAG:", llm.invoke(question))

The model gives a plausible but incorrect answer (hallucination). RAG fixes this by enforcing a contract: **the model may only answer using evidence we provide.**

## 2. Ingesting Enterprise Data

Let's load some mock internal documents into an in-memory vector store.

In [ ]:
documents = [
    Document(page_content="NovaTech's Q2 2025 financial review: Cloud infrastructure costs increased by 14% due to the new cluster deployment.", metadata={"source": "finance_review_q2.md", "tenant": "finance"}),
    Document(page_content="HR Policy: Parental leave has been extended to 16 weeks globally.", metadata={"source": "hr_policy_v2.md", "tenant": "hr"}),
    Document(page_content="Runbook 17: For checkout errors, correlate the 08:42 deployment with payment dependency latency before proposing rollback.", metadata={"source": "runbooks/checkout.md", "tenant": "engineering"})
]

# We use FakeEmbeddings for demonstration. In production, use OpenAIEmbeddings() or similar.
embeddings = FakeEmbeddings(size=1536)
vectorstore = InMemoryVectorStore.from_documents(documents, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 1})

## 3. The Grounded RAG Pipeline

Now we define the strict prompt and connect the retriever.

In [ ]:
template = """
You are a helpful NovaTech enterprise assistant.
Answer the question based ONLY on the following context. 
If you cannot answer the question based on the context, say "I do not know based on the provided evidence."

Context:
{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

# Update our fake LLM to simulate the correct grounded response
llm = FakeListLLM(responses=["Based on the finance review, cloud infrastructure costs increased by 14% due to the new cluster deployment."])

def format_docs(docs):
    return "\n\n".join(f"[Source: {d.metadata['source']}]\n{d.page_content}" for d in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("Model answer WITH RAG:")
print(rag_chain.invoke(question))

## Reflection

1. **Traceability:** Look at `format_docs`. Why is it important to inject the `[Source: ...]` metadata into the LLM's context window?
2. **Abstention:** What happens if the context is empty? (The prompt instruction forces it to say 'I do not know').

In the next lab, we will replace the `InMemoryVectorStore` and `FakeEmbeddings` with a real local vector database and embedding model.